# Dense Face Mesh with UniFace

<div style="display:flex; flex-wrap:wrap; align-items:center;">
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pepy.tech/projects/uniface"><img alt="PyPI Downloads" src="https://static.pepy.tech/personalized-badge/uniface?period=total&units=international_system&left_color=grey&right_color=blue&left_text=Downloads"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pypi.org/project/uniface/"><img alt="PyPI Version" src="https://img.shields.io/pypi/v/uniface.svg"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://opensource.org/licenses/MIT"><img alt="License" src="https://img.shields.io/badge/License-MIT-blue.svg"></a>
  <a style="margin-bottom:6px;" href="https://github.com/yakhyo/uniface"><img alt="GitHub Stars" src="https://img.shields.io/github/stars/yakhyo/uniface.svg?style=social"></a>
</div>

**UniFace** is a lightweight, production-ready Python library for face detection, recognition, tracking, landmark analysis, face parsing, gaze estimation, and face attributes.

🔗 **GitHub**: [github.com/yakhyo/uniface](https://github.com/yakhyo/uniface) | 📚 **Docs**: [yakhyo.github.io/uniface](https://yakhyo.github.io/uniface)

---

This notebook demonstrates **MediaPipe Face Mesh** — 468 dense 3D landmarks per face.

Unlike the other landmark models it returns **three** coordinates per point and a face-presence score, and it runs every face in an image through a **single batched inference call**.

## 1. Install UniFace

In [ ]:
%pip install -q "uniface[cpu]"

# Clone repo for assets (Colab only)
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.exists('uniface'):
        !git clone --depth 1 https://github.com/yakhyo/uniface.git
    os.chdir('uniface/examples')

## 2. Import Libraries

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import uniface
from uniface.detection import SCRFD, BlazeFace
from uniface.draw import draw_mesh
from uniface.landmark import FaceMesh

print(f"UniFace version: {uniface.__version__}")

## 3. Initialize Models

In [ ]:
# Any detector works — Face Mesh only needs a box plus the two eye points
detector = SCRFD()
mesher = FaceMesh()

print(f"Landmarks: {mesher.num_landmarks}")
print(f"Input size: {mesher.input_size}x{mesher.input_size}")

## 4. Detect and Mesh

Face Mesh needs a bounding box plus the first two landmarks (the eyes), which it uses to rotate the crop so the eye line is horizontal. Passing `Face` objects does this automatically.

In [ ]:
image = cv2.imread('../assets/einstein.png')

faces = detector.detect(image)
print(f"Detected {len(faces)} face(s)")

# One batched call for every face in the image
results = mesher.predict(image, faces)

result = results[0]
print(f"Landmarks shape: {result.landmarks.shape}")   # (468, 3)
print(f"2D only:         {result.points_2d.shape}")   # (468, 2)
print(f"Presence score:  {result.score:.4f}")

## 5. Visualize

`draw_mesh` has three render modes. `full` draws the dense 2556-edge tessellation — the most detailed, but noticeably slower. Prefer `partial` or `points` for video.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, mode in zip(axes, ['points', 'partial', 'full']):
    canvas = image.copy()
    draw_mesh(canvas, result.landmarks, mode=mode)
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(f"mode='{mode}'")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. The Depth Coordinate

`z` is **relative** depth on the same pixel scale as `x`/`y`, where smaller is closer to the camera. It has no absolute origin, so it is only meaningful *within* one face — not between faces or between images.

In [ ]:
points = result.landmarks

plt.figure(figsize=(7, 7))
scatter = plt.scatter(points[:, 0], points[:, 1], c=points[:, 2], cmap='viridis', s=8)
plt.colorbar(scatter, label='z (relative depth, smaller = closer)')
plt.gca().invert_yaxis()
plt.gca().set_aspect('equal')
plt.title('468 landmarks coloured by depth')
plt.show()

print(f"z range: {points[:, 2].min():.1f} .. {points[:, 2].max():.1f}")
print("The nose tip is the closest point; the ears and jaw edges sit furthest back.")

## 7. MediaPipe Parity

Seeding the mesh with **BlazeFace** reproduces MediaPipe's own pipeline exactly.

> ⚠️ BlazeFace returns **6** MediaPipe keypoints whose 4th point is a mouth *center*, not corners, so they cannot be fitted to the 5-point alignment template. It declares this with `supports_alignment = False`, and `FaceAnalyzer` disables recognition for it rather than producing broken embeddings.

In [ ]:
# BlazeFace is the detector mp.solutions.face_mesh runs internally
blazeface = BlazeFace()

bf_faces = blazeface.detect(image)
bf_results = mesher.predict(image, bf_faces)

print(f"BlazeFace keypoints: {bf_faces[0].landmarks.shape}")       # (6, 2), not (5, 2)
print(f"supports_alignment:  {blazeface.supports_alignment}")      # False

canvas = image.copy()
draw_mesh(canvas, bf_results[0].landmarks, mode='full')

plt.figure(figsize=(7, 7))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('MediaPipe parity: BlazeFace + Face Mesh')
plt.axis('off')
plt.show()

## 8. Other Ways to Call It

In [ ]:
# Face Mesh implements the same interface as Landmark106 and PIPNet
landmarks_2d = mesher.get_landmarks(image, faces[0].bbox)
print(f"get_landmarks: {landmarks_2d.shape}")

# Without a detector, pass boxes directly
x1, y1, x2, y2 = faces[0].bbox
manual = mesher.predict(image, bboxes=[[x1, y1, x2, y2]])
print(f"From a raw box: {manual[0].landmarks.shape}")

# Tune the crop with margin= if the mesh clips on unusual framing
tight = mesher.predict(image, faces, margin=0.1)[0]
print(f"Tighter crop shifts the fit: {not np.allclose(tight.landmarks, result.landmarks)}")

## Summary

| | |
|---|---|
| **Output** | `(468, 3)` — `x`/`y` in image pixels, `z` relative depth on the same scale |
| **Presence score** | Saturates near 1.0; confirms the model ran, not a discriminative confidence |
| **Detectors** | Works with any of them — needs a box plus the two eye points |
| **Batching** | `predict()` runs all faces in one inference call |
| **Drop-in** | `get_landmarks()` matches `Landmark106` / `PIPNet` |

### Next steps

- [Landmarks documentation](https://yakhyo.github.io/uniface/modules/landmarks/)
- [Detection documentation](https://yakhyo.github.io/uniface/modules/detection/) — BlazeFace and the 6-keypoint layout
- [Coordinate systems](https://yakhyo.github.io/uniface/concepts/coordinate-systems/) — what `z` means

> Face Mesh and BlazeFace weights originate from Google MediaPipe and are Apache-2.0 licensed.
> See [Licenses & Attribution](https://yakhyo.github.io/uniface/license-attribution/).